# Neural networks, built from matrix multiplies

> A hidden layer and one nonlinearity is the whole idea. Build the forward pass by hand, train it by brute force, and discover exactly why backpropagation had to be invented.

Read this chapter at `/learn/08-neural-networks/`. Exported from `src/content/chapters/08-neural-networks.mdx` — edit there, not here.


A neural network is a linear model, then a squashing function, then another
linear model.

That's it. That's the architectural idea, and I want you to have it in one
sentence before anybody shows you a diagram with circles and arrows, because the
diagrams make it look mysterious and it isn't.

Everything else you'll ever hear about — convolutions, attention, residual
connections, normalisation layers — is a constraint bolted onto that sandwich.

Today you build one out of nothing but `@` and
arrays.

## First, the wall a linear model hits

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X, y = make_moons(n_samples=800, noise=0.22, random_state=0)
X = (X - X.mean(0)) / X.std(0)                    # standardise, always
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

lin = LogisticRegression().fit(X_tr, y_tr)
print(f"logistic regression, validation accuracy: {lin.score(X_va, y_va):.3f}")

Eighty-something percent — and no amount of training will improve it. Not more
steps, not a better learning rate, not more data.

This is worth being precise about, because it's a distinction that will save you
a lot of confused debugging. This is **not an optimisation failure**. The
optimiser did its job perfectly and found the best straight line there is.

It's a **representation failure**. The function you need isn't in the family you
chose. You asked for the best line, you got the best line, and the best line
isn't good enough.

## The obvious fix, and why it fails

Fine — stack two linear layers. More layers, more power. Right?

In [ ]:
rng = np.random.default_rng(0)
W1, W2 = rng.normal(size=(2, 8)), rng.normal(size=(8, 1))

x = rng.normal(size=(4, 2))
two_layers = (x @ W1) @ W2
one_layer  = x @ (W1 @ W2)
print("identical:", np.allclose(two_layers, one_layer))
print("the 'deep' model is really just this 2x1 matrix:\n", (W1 @ W2).round(3).T)

Ah.

Matrix multiplication is *associative*, so a composition of linear maps is itself
a linear map. Your "two-layer network" is one matrix in a costume.

And this doesn't get better with more layers. A hundred stacked linear layers is
still a straight line — with a hundred times the parameters, a hundred times the
compute, and precisely none of the power.

**The nonlinearity is not a detail. It's the entire reason depth means anything.**

Without it, depth is a very expensive way to compute a single matrix. With it,
depth is the most powerful idea in the field. That's a lot resting on one
`max(0, z)`.

## One nonlinearity changes everything

In [ ]:
relu = lambda z: np.maximum(0, z)

z = np.linspace(-3, 3, 200)
plt.figure(figsize=(4.6, 2.4))
plt.plot(z, relu(z)); plt.title("ReLU(z) = max(0, z)")
plt.axhline(0, c="grey", lw=.5); plt.axvline(0, c="grey", lw=.5)
plt.tight_layout()

ReLU is `max(0, z)`. Negative numbers become zero, positive
numbers are left alone.

I know. It looks far too simple to be the thing that unlocked deep learning, and
that reaction is completely reasonable. So here's why it matters, and it's a
lovely bit of reasoning.

Its **derivative is exactly 1** for positive inputs. Not 0.9, not 0.99 — exactly
1. So when a gradient passes back through a ReLU, it comes out the same size it
went in. Stack thirty of them and the gradient survives.

Now compare sigmoid, whose derivative *peaks* at 0.25 and is
usually much smaller. Thirty layers of that multiplies your gradient by something
like $10^{-18}$. It arrives at the first layer as a rounding error, and the first
layer learns nothing, ever.

That's the vanishing gradient problem, and `max(0, z)` is most of the fix. A
field-changing improvement that a beginner could have written.

Right — the network:

In [ ]:
def init(n_in, n_hidden, n_out, seed=0):
    rng = np.random.default_rng(seed)
    # He initialisation: variance 2/n_in keeps activations from shrinking layer
    # to layer. The 2 is because ReLU discards half the distribution.
    return {
        "W1": rng.normal(0, np.sqrt(2 / n_in), (n_in, n_hidden)),
        "b1": np.zeros(n_hidden),
        "W2": rng.normal(0, np.sqrt(2 / n_hidden), (n_hidden, n_out)),
        "b2": np.zeros(n_out),
    }

def forward(p, X):
    z1 = X @ p["W1"] + p["b1"]      # (n, hidden)   linear
    a1 = np.maximum(0, z1)          # (n, hidden)   nonlinear
    z2 = a1 @ p["W2"] + p["b2"]     # (n, 1)        linear
    return 1 / (1 + np.exp(-z2))    # (n, 1)        probability

params = init(2, 16, 1)
forward(params, X_tr[:3]).ravel().round(3)

Five lines. Two matrix multiplies, two
broadcast bias additions, one `maximum`.

That's a neural network. Every network in this book — and, with more layers and
some extra structure, every network anywhere — is that, repeated.

`params` is a `HashMap<&str, Array2<f64>>` because that's the idiom here; you'd
write a struct with named fields and be happier.

The shapes *are* the type signature, and they're the thing worth checking:

```
X   (n, 2)  @  W1 (2, 16)  ->  (n, 16)  + b1 (16,)   broadcast
    (n, 16) @  W2 (16, 1)  ->  (n, 1)   + b2 (1,)    broadcast
```

The inner dimensions cancel; the outer ones survive. When a network throws a
shape error — and it will, today — this is the arithmetic to do on paper before
you touch the code.

## Training it, the honest slow way

We have a model and a loss. We need gradients.

We haven't derived them yet, so let's get them the brute-force way: nudge each
parameter a tiny bit, and see what the loss does.

In [ ]:
def loss(p, X, y):
    prob = forward(p, X).ravel()
    prob = np.clip(prob, 1e-9, 1 - 1e-9)
    return -(y * np.log(prob) + (1 - y) * np.log(1 - prob)).mean()

def numerical_grad(p, X, y, eps=1e-5):
    """The definition of a derivative, applied 2x per parameter."""
    grads = {}
    for key, mat in p.items():
        g = np.zeros_like(mat)
        for idx in np.ndindex(mat.shape):
            original = mat[idx]
            mat[idx] = original + eps; hi = loss(p, X, y)
            mat[idx] = original - eps; lo = loss(p, X, y)
            mat[idx] = original
            g[idx] = (hi - lo) / (2 * eps)
        grads[key] = g
    return grads

small = init(2, 4, 1)
g = numerical_grad(small, X_tr[:64], y_tr[:64])
print("dL/dW1 =\n", g["W1"].round(4))

This is the literal definition of a derivative, and it is
completely, unimpeachably correct.

It is also useless. And I think it's worth measuring *exactly* how useless,
because the number is the reason the next chapter exists.

In [ ]:
import time

for hidden in [4, 16, 64]:
    p = init(2, hidden, 1)
    n_params = sum(v.size for v in p.values())
    t = time.perf_counter()
    numerical_grad(p, X_tr[:64], y_tr[:64])
    dt = time.perf_counter() - t
    print(f"hidden={hidden:3d}  {n_params:5d} params  "
          f"{2 * n_params:6d} forward passes  {dt * 1000:8.1f} ms per gradient")

**Two forward passes per parameter.** Linear in the parameter count, and the
constant is a whole forward pass.

Let's extrapolate, because the numbers are genuinely startling.

A small vision model has 10 million parameters. So one gradient step needs 20
million forward passes. At a millisecond each, that's **five and a half hours**
— for *one step*, out of the tens of thousands a model needs.

A modern language model has around $10^{11}$ parameters. Numerical
differentiation isn't slow here. It's arithmetically impossible, by dozens of
orders of magnitude. You could run it until the heat death of the universe and
not finish one step.

Now here's tomorrow's chapter in one sentence:

**Backpropagation computes the entire gradient — every parameter, all of them —
for roughly the cost of one extra forward pass.**

Not one per parameter. One. Total.

That is the single fact this whole field is built on. Sit with the size of the
gap for a moment: from "five and a half hours per step" to "about twice a forward
pass," and it's the same answer, exactly.

## But it does work

Small enough network, patient enough, and brute force does train it:

In [ ]:
p = init(2, 8, 1, seed=1)
sub_X, sub_y = X_tr[:200], y_tr[:200]
history = []
for step in range(60):
    grads = numerical_grad(p, sub_X, sub_y)
    for k in p:
        p[k] -= 0.5 * grads[k]
    history.append(loss(p, sub_X, sub_y))

acc = ((forward(p, X_va).ravel() > 0.5).astype(int) == y_va).mean()
print(f"after 60 brute-force steps: loss {history[-1]:.4f}, validation accuracy {acc:.3f}")

In [ ]:
xx, yy = np.meshgrid(np.linspace(-2.2, 2.4, 220), np.linspace(-2.2, 2.4, 220))
zz = forward(p, np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(1, 2, figsize=(8.6, 3.2))
ax[0].plot(history); ax[0].set_xlabel("step"); ax[0].set_ylabel("loss")
ax[0].set_title("60 steps, each costing 2x145 forward passes")
ax[1].contourf(xx, yy, zz, levels=20, cmap="coolwarm", alpha=.7)
ax[1].scatter(X_va[:, 0], X_va[:, 1], c=y_va, s=8, cmap="coolwarm", edgecolors="none")
ax[1].set_title("a curved boundary"); ax[1].set_xticks([]); ax[1].set_yticks([])
plt.tight_layout()

**A curve.** Not a line.

Eight hidden units and one call to `maximum` bought a decision boundary that
logistic regression could never express, no matter what you did to it. And that's
after sixty steps of the slowest gradient method in existence.

It's barely ahead of the straight line so far — sixty steps is nothing. Give it a
real optimiser and it reaches 0.96, which the last cell of this chapter will
show you.

## Why a hidden layer can express anything

In [ ]:
grid = np.linspace(-3, 3, 400)
rng2 = np.random.default_rng(4)
W1_, b1_ = rng2.normal(size=(1, 12)) * 3, rng2.normal(size=12) * 2
W2_ = rng2.normal(size=(12, 1))
hidden = np.maximum(0, grid.reshape(-1, 1) @ W1_ + b1_)

fig, ax = plt.subplots(1, 2, figsize=(8.6, 2.9))
ax[0].plot(grid, hidden, lw=.8); ax[0].set_title("12 ReLU units — each a hinge")
ax[1].plot(grid, hidden @ W2_, c="crimson"); ax[1].set_title("their weighted sum")
plt.tight_layout()

Look at the left panel. Every hidden unit is a **hinge** — flat until its
threshold, then a straight ramp. Twelve hinges, at twelve different places, with
twelve different slopes.

Now the right panel: their weighted sum. A wiggly curve, made entirely of
straight pieces.

And that's the intuition for one of the famous results in the field. Enough
hinges, placed and scaled freely, can approximate any continuous function you
like, to any accuracy you like. It's the same reason you can draw any curve with
enough short straight lines.

That statement is the **universal approximation theorem** (Cybenko 1989, Hornik
1991) — and it's worth knowing exactly how little it actually promises.

It says a wide enough single hidden layer *can represent* any continuous function
on a bounded domain.

It does not say how wide. The required width can be exponential in the input
dimension. It does not say you can *find* the right weights — only that they
exist. And it says precisely nothing about generalisation.

So it's a statement about the existence of a needle, in a haystack of unstated
size, with no method given for finding it. It gets quoted far more often than it
gets used.

The practically important fact is different, and it's empirical rather than
proved: **depth is exponentially more efficient than width.** Functions that need
$2^n$ units in one layer often need only $O(n)$ units spread across $n$ layers.

That's why the field went deep instead of wide. Not because of a theorem —
because of what kept working.

## Width and depth, empirically

In [ ]:
def train(hidden_sizes, steps=400, lr=0.5, seed=0):
    """Analytic gradients — the derivation is tomorrow. Here just to compare shapes."""
    rng = np.random.default_rng(seed)
    sizes = [2, *hidden_sizes, 1]
    Ws = [rng.normal(0, np.sqrt(2 / a), (a, b)) for a, b in zip(sizes, sizes[1:])]
    bs = [np.zeros(b) for b in sizes[1:]]
    for _ in range(steps):
        acts, a = [X_tr], X_tr
        for i, (W, b) in enumerate(zip(Ws, bs)):
            z = a @ W + b
            a = np.maximum(0, z) if i < len(Ws) - 1 else 1 / (1 + np.exp(-z))
            acts.append(a)
        delta = (acts[-1].ravel() - y_tr).reshape(-1, 1) / len(y_tr)
        for i in range(len(Ws) - 1, -1, -1):
            gW, gb = acts[i].T @ delta, delta.sum(0)
            if i > 0:
                delta = (delta @ Ws[i].T) * (acts[i] > 0)
            Ws[i] -= lr * gW; bs[i] -= lr * gb
    a = X_va
    for i, (W, b) in enumerate(zip(Ws, bs)):
        z = a @ W + b
        a = np.maximum(0, z) if i < len(Ws) - 1 else 1 / (1 + np.exp(-z))
    return ((a.ravel() > 0.5).astype(int) == y_va).mean()

for shape in [[2], [8], [32], [128], [16, 16], [16, 16, 16]]:
    print(f"hidden {str(shape):16s}  validation accuracy {train(shape):.3f}")

Read that table carefully, because it's more honest than most of what you'll see
about model size.

Two hidden units gives you the logistic regression number back almost exactly —
too little capacity to bend at all. Eight barely improves on it.

Thirty-two reaches 0.96. And then... nothing. 128 units buys you nothing. Two
layers buys nothing. Three layers buys nothing.

The task is easy, and we've hit the noise floor. There is no more signal in this
data to extract, and every extra parameter is just a parameter.

**Capacity only helps while capacity is the constraint.** Past that point, more
parameters cost you compute, memory and overfitting risk in exchange for exactly
zero.

That's worth having felt once, before the instinct to reach for a bigger model
sets in.

**"Why is `b1` initialised to zeros but `W1` random?"** Only the weights need
symmetry broken — question 2 below shows why. Biases can safely start at zero
because the weights already make each unit different.

**"What is `np.ndindex` doing?"** It yields every index tuple of an array's
shape, so the loop visits every single element regardless of dimensionality. It's
how `numerical_grad` stays generic over a `(2,16)` and a `(16,1)`.

**"Why `(hi - lo) / (2 * eps)` and not `(hi - loss) / eps`?"** The two-sided
version is more accurate — errors cancel rather than accumulate. It costs twice
as much, which for a method this slow hardly matters.

**"My network trains to exactly 0.5 accuracy and never moves."** Almost always
dead ReLUs or a bad initialisation scale. Print the mean activation of your
hidden layer; if it's zero, every unit has died and there's no gradient anywhere.
Exercise 3 below reproduces this deliberately.

**"The `train` function has backpropagation in it and you haven't taught it
yet!"** Guilty — it's there so you can compare *shapes* today without waiting.
Tomorrow you'll derive every line of it, and you're very welcome to come back and
read it then. It'll be about six lines you fully understand.

In [ ]:
# 1. Replace ReLU with the identity in `forward`. Retrain with `train([16, 16])`
#    modified accordingly. What accuracy do you get, and why exactly that number?
#
# 2. Set every weight in `init` to zero instead of random. Train. What happens
#    to the hidden units, and why? (This is called the symmetry problem.)
#
# 3. Multiply the He initialisation by 50. Watch the first-layer activations.
#
# 4. How many parameters does a [2, 128, 1] network have? Count by hand,
#    then verify.

print("replace me")

For 1, you already saw the answer earlier in this chapter — look at the "two
linear layers are one linear layer" cell and predict the number before running.

For 2, think about what happens to two hidden units that start *identical* and
then receive *identical* gradients. What could ever make them different?

In [ ]:
# 2. All-zero initialisation
def forward_zero(X, hidden=8, steps=200, lr=0.5):
    W1, b1 = np.zeros((2, hidden)), np.zeros(hidden)
    W2, b2 = np.zeros((hidden, 1)), np.zeros(1)
    for _ in range(steps):
        a1 = np.maximum(0, X @ W1 + b1)
        out = 1 / (1 + np.exp(-(a1 @ W2 + b2)))
        d2 = (out.ravel() - y_tr).reshape(-1, 1) / len(y_tr)
        d1 = (d2 @ W2.T) * (a1 > 0)
        W2 -= lr * a1.T @ d2; b2 -= lr * d2.sum(0)
        W1 -= lr * X.T @ d1;  b1 -= lr * d1.sum(0)
    return W1

W1_zero = forward_zero(X_tr)
print("all hidden units identical:", np.allclose(W1_zero, W1_zero[:, :1]))
print("first three columns:\n", W1_zero[:, :3].round(6))

# 3. Initialisation scale
for scale in [1, 10, 50]:
    p = init(2, 64, 1); p["W1"] *= scale
    a1 = np.maximum(0, X_tr @ p["W1"] + p["b1"])
    print(f"scale x{scale:3d}   mean activation {a1.mean():9.3f}   "
          f"dead units {(a1.max(0) == 0).sum():2d}/64")

# 4. Parameter count
p = init(2, 128, 1)
print("\nby hand: 2*128 + 128 + 128*1 + 1 =", 2*128 + 128 + 128*1 + 1)
print("actual  :", sum(v.size for v in p.values()))

**Question 1** hands you the logistic regression number back — around 0.85. Drop
the nonlinearity and the whole composition collapses to a single matrix, exactly
as the second cell of this chapter showed. All that depth simply evaporates.

**Question 2** is the **symmetry problem**, and it's the reason nobody ever
initialises to zero.

Every hidden unit starts identical. So every unit computes the same thing. So
every unit receives an identical gradient. So every unit updates identically —
and stays identical, forever. A 64-unit layer behaves exactly like a 1-unit layer
for the entire training run, and nothing can ever break the tie.

Random initialisation exists to break that symmetry. That is its *whole job*, and
it's why He and Xavier initialisation specify a **variance** rather than a value.
Nobody cares what the numbers are, only that they're different.

**Question 3** shows the opposite failure. Too large an initialisation and ReLU
units saturate on one side; some **die** entirely — always negative input, always
zero output, therefore zero gradient, therefore they never recover. Watch the
"dead units" count climb.

He initialisation's $\sqrt{2/n_{in}}$ is chosen precisely so activation variance
is preserved layer to layer, and the 2 is there specifically because ReLU throws
away half the distribution.

**Question 4** is 385 parameters. The formula for a dense layer is
`in * out + out` — one weight per connection, one bias per output.

Being able to count parameters from an architecture description is a genuinely
useful skill. It tells you your memory cost, and it's the first sanity check to
run when a paper's numbers look odd.

Tomorrow: how to get every one of those gradients for the price of one forward
pass. It's my favourite chapter.